# PTB-XL — CNN-BiLSTM-Attention versus CNN-Transformer comparison

This notebook compares independently trained ECG model artifacts. The original ECG evaluation notebooks remain unchanged. All comparisons must use the same PTB-XL patient-disjoint test records, labels and preprocessing version. If the recorded metadata do not prove identical splits and target columns, the comparison is flagged rather than silently accepted.

In [1]:
from pathlib import Path
import json, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def find_root(start=None):
    current=Path(start or Path.cwd()).resolve()
    for p in [current,*current.parents]:
        if (p/"src").is_dir() and (p/"config").is_dir(): return p
        if (p/"ml-service"/"src").is_dir(): return p/"ml-service"
    raise FileNotFoundError("Could not locate ml-service.")
ML_ROOT=find_root()
BASE=ML_ROOT/"artifacts"/"metrics"/"ecg"
MODEL_FILES={
    "CNN-BiLSTM-Attention": BASE/"cnn_bilstm_attention"/"test_metrics.json",
    "CNN-Temporal-Transformer": BASE/"ecg_transformer"/"metrics.json",
}

## 1. Load model-level metrics and verify provenance

In [2]:
records=[]
for model_name,path in MODEL_FILES.items():
    if not path.is_file():
        print(f"[MISSING] {model_name}: {path}")
        continue
    payload=json.loads(path.read_text(encoding="utf-8"))
    records.append({"model":model_name,"path":str(path),"payload":payload})
if len(records)<2:
    print("Both model artifacts are required for a direct comparative table. Train/evaluate the missing model first.")

## 2. Compare only metrics with matching definitions

The legacy CNN-BiLSTM notebook stores one aggregate metric dictionary per task group. The CNN-Transformer notebook stores metrics per label. This cell reports both faithfully: legacy aggregate values remain labelled `reported_aggregate`, while Transformer per-label values and their unweighted macro means remain explicitly labelled `per_label` and `macro_mean`. These are not automatically interchangeable. Do not claim a direct numerical winner unless the two experiments are recomputed with the same patient split, label masks, threshold policy and aggregation definition.


In [3]:
summary_rows=[]
label_rows=[]
for item in records:
    payload=item["payload"]
    if item["model"]=="CNN-BiLSTM-Attention":
        groups={"diagnostic":payload.get("diagnostic_test",{}),"rhythm":payload.get("rhythm_test",{})}
        for group_name,values in groups.items():
            if isinstance(values,dict) and values and all(not isinstance(v,dict) for v in values.values()):
                summary_rows.append({"model":item["model"],"group":group_name,"aggregation":"reported_aggregate",**values})
            elif isinstance(values,dict):
                for label,metric in values.items():
                    if isinstance(metric,dict): label_rows.append({"model":item["model"],"group":group_name,"label":label,**metric})
    else:
        groups=payload.get("test",{})
        for group_name,values in groups.items():
            if not isinstance(values,dict): continue
            current=[]
            for label,metric in values.items():
                if not isinstance(metric,dict): continue
                row={"model":item["model"],"group":group_name,"label":label,**metric}
                label_rows.append(row); current.append(metric)
            numeric_keys=[key for key in ("auroc","auprc","brier","f1_at_0_5","precision_at_0_5","recall_at_0_5") if current and all(isinstance(m.get(key),(int,float)) and np.isfinite(m[key]) for m in current)]
            if numeric_keys:
                summary_rows.append({"model":item["model"],"group":group_name,"aggregation":"macro_mean",**{key:float(np.mean([m[key] for m in current])) for key in numeric_keys},"n_labels":len(current)})

summary=pd.DataFrame(summary_rows)
label_comparison=pd.DataFrame(label_rows)
print("Task-group summary (aggregation method is shown explicitly):")
display(summary)
print("Per-label rows (available for models whose artifacts save them):")
display(label_comparison)
summary.to_csv(BASE/"model_comparison_summary.csv",index=False)
label_comparison.to_csv(BASE/"model_comparison_per_label.csv",index=False)
print("Reminder: reported aggregate and macro-mean values are different estimands; recompute both models identically before ranking them.")


Task-group summary (aggregation method is shown explicitly):


,model,group,aggregation,auroc,auprc,f1,precision,recall,micro_f1,brier,f1_at_0_5,precision_at_0_5,recall_at_0_5,n_labels
0,CNN-BiLSTM-Attention,diagnostic,reported_aggregate,0.925416,0.815309,0.742128,0.751426,0.73874,0.773941,NaN,NaN,NaN,NaN,NaN
1,CNN-BiLSTM-Attention,rhythm,reported_aggregate,0.921660,0.583981,0.550995,0.598687,0.63793,0.849799,NaN,NaN,NaN,NaN,NaN
2,CNN-Temporal-Transformer,diagnostic,macro_mean,0.915233,0.792796,NaN,NaN,NaN,NaN,0.091244,0.706133,0.791221,0.661008,5.0
3,CNN-Temporal-Transformer,rhythm,macro_mean,0.886297,0.408882,NaN,NaN,NaN,NaN,0.025928,0.344942,0.445907,0.310745,9.0


Per-label rows (available for models whose artifacts save them):


,model,group,label,status,n,auroc,auprc,brier,f1_at_0_5,precision_at_0_5,recall_at_0_5
0,CNN-Temporal-Transformer,diagnostic,NORM,computed,2203,0.943319,0.920549,0.105359,0.850070,0.773152,0.943983
1,CNN-Temporal-Transformer,diagnostic,MI,computed,2203,0.898496,0.778889,0.114385,0.589623,0.847458,0.452080
2,CNN-Temporal-Transformer,diagnostic,STTC,computed,2203,0.932655,0.809994,0.083507,0.765336,0.779762,0.751434
3,CNN-Temporal-Transformer,diagnostic,CD,computed,2203,0.904365,0.800831,0.087145,0.710900,0.867052,0.602410
4,CNN-Temporal-Transformer,diagnostic,HYP,computed,2203,0.897333,0.653716,0.065824,0.614737,0.688679,0.555133
5,CNN-Temporal-Transformer,rhythm,RHYTHM_SR,computed,2203,0.833966,0.920751,0.116373,0.908788,0.851929,0.973778
6,CNN-Temporal-Transformer,rhythm,RHYTHM_AFIB,computed,2203,0.969864,0.752221,0.027779,0.722973,0.743056,0.703947
7,CNN-Temporal-Transformer,rhythm,RHYTHM_AFLT,computed,2203,0.928311,0.118432,0.003004,0.000000,0.000000,0.000000
8,CNN-Temporal-Transformer,rhythm,RHYTHM_STACH,computed,2203,0.983878,0.730559,0.019145,0.571429,0.818182,0.439024
9,CNN-Temporal-Transformer,rhythm,RHYTHM_SBRAD,computed,2203,0.926470,0.324882,0.023112,0.162162,0.600000,0.093750


Reminder: reported aggregate and macro-mean values are different estimands; recompute both models identically before ranking them.


## 3. Fair-comparison checklist

Before claiming one model is better, verify:

1. The same manifest version and patient-level train/validation/test partition were used.
2. The same diagnostic and rhythm target columns were evaluated.
3. Metrics are calculated on the same held-out patient records.
4. The same threshold-selection procedure was used, with thresholds selected on validation data only.
5. AUROC, AUPRC, sensitivity, specificity, F1 and calibration are reported per label.
6. Confidence intervals are calculated using patient-level bootstrap resampling if multiple ECG records per patient are present.

If these conditions are not verified, this notebook provides an artifact inventory, not a valid head-to-head performance claim.

In [4]:
# Optional exact record-level comparison when both prediction files exist.
prediction_files={
    "CNN-BiLSTM-Attention": BASE/"cnn_bilstm_attention"/"ptbxl_test_predictions.csv",
    "CNN-Temporal-Transformer": BASE/"ecg_transformer"/"ptbxl_test_predictions.csv",
}
for model_name,path in prediction_files.items():
    print(model_name, "prediction export exists:", path.is_file(), "->", path)

CNN-BiLSTM-Attention prediction export exists: True -> D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\artifacts\metrics\ecg\cnn_bilstm_attention\ptbxl_test_predictions.csv
CNN-Temporal-Transformer prediction export exists: True -> D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\artifacts\metrics\ecg\ecg_transformer\ptbxl_test_predictions.csv
